# AI & ML Practical Exercise — Practical 2
## Bayesian Network for Medical Diagnosis (Heart Disease Dataset)

**Course:** Sub : 704 : AI_ML Practical Exercise  
**Problem Statement:**  
> Write a program to construct a Bayesian network considering medical data. Use this model to demonstrate the diagnosis of heart patients using standard Heart Disease Data Set. Calculate the accuracy, precision, and recall for your data set. (dataset: `heart.csv`)

---
### Pipeline Overview:
1. **Environment Setup & Imports**: Load `pgmpy`, `sklearn`, `pandas`, `numpy`, and visualization tools.
2. **Data Loading & Exploration**: Read `heart.csv` and inspect features.
3. **Data Preprocessing & Discretization**: Encode categorical variables and discretize continuous variables into quantile bins.
4. **Train-Test Split**: Stratified splitting into 80% training and 20% testing sets.
5. **Bayesian Network Construction & DAG Visualization**: Build the directed acyclic graph (DAG).
6. **Parameter Estimation (CPD Learning)**: Learn Conditional Probability Distributions using Bayesian Estimator with BDeu prior.
7. **Demonstration of Heart Patient Diagnosis**: Infer disease probability on specific clinical patient profiles.
8. **Model Evaluation**: Compute Accuracy, Precision, Recall, Confusion Matrix, and Classification Report on test data.

### Step 1: Import Required Libraries
We import `pandas` and `numpy` for data manipulation, `sklearn` for dataset splitting and metric evaluations, and `pgmpy` for Bayesian Network construction, parameter learning, and probabilistic inference.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

# Ensure compatibility with both newer and older pgmpy versions
try:
    from pgmpy.models import DiscreteBayesianNetwork
except ImportError:
    from pgmpy.models import BayesianNetwork as DiscreteBayesianNetwork

from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

print("Libraries successfully imported!")

KeyboardInterrupt: 

### Step 2: Load and Inspect the Dataset (`heart.csv`)
The dataset contains clinical and pathological attributes of patients. We inspect the structure, columns, and sample records.

In [ ]:
# Load dataset
df = pd.read_csv("heart.csv")

print("Dataset Shape:", df.shape)
print("\nColumns in Dataset:")
print(df.columns.tolist())

# Preview first 5 rows
df.head()

### Step 3: Data Preprocessing & Discretization
Bayesian Networks with discrete nodes require categorical/discrete states for all variables.
- Drop index column (`Unnamed: 0`).
- Map target variable `AHD` (`No` -> 0, `Yes` -> 1).
- Encode categorical variable `ChestPain` into numeric codes.
- Discretize continuous variables (`Age`, `MaxHR`, `Oldpeak`) into 3 quantiles (Low = 0, Medium = 1, High = 2) using `pd.qcut`.
- Handle missing values with `.dropna()`.

In [ ]:
# 1. Drop irrelevant index column if present
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# 2. Map target AHD (Atherosclerotic Heart Disease) to binary 0/1
df["AHD"] = df["AHD"].map({"No": 0, "Yes": 1})

# 3. Select key diagnostic features
features = ["Age", "Sex", "ChestPain", "MaxHR", "ExAng", "Oldpeak", "AHD"]
data = df[features].copy()

# 4. Drop any missing values to ensure clean CPD learning
data = data.dropna()

# 5. Encode categorical feature 'ChestPain'
# Categories: asymptomatic, nonanginal, nontypical, typical
data["ChestPain"] = data["ChestPain"].astype("category").cat.codes

# 6. Discretize continuous numerical features into 3 quantile bins (Low=0, Medium=1, High=2)
for col in ["Age", "MaxHR", "Oldpeak"]:
    data[col] = pd.qcut(data[col], q=3, labels=False, duplicates="drop")

print("Preprocessed Data Shape:", data.shape)
print("\nFirst 5 preprocessed records:")
data.head()

NameError: name 'df' is not defined

### Step 4: Split into Training and Testing Sets
We split the dataset into **80% training data** (used to learn the Conditional Probability Distributions) and **20% testing data** (used to evaluate diagnostic accuracy, precision, and recall). We use stratified sampling to preserve the target class balance.

In [ ]:
train, test = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["AHD"]
)

print(f"Training set samples: {train.shape[0]} ({train.shape[1]} features)")
print(f"Testing set samples:  {test.shape[0]} ({test.shape[1]} features)")
print("\nTraining class distribution (AHD):")
print(train["AHD"].value_counts(normalize=True))

### Step 5: Construct the Bayesian Network (DAG Structure)
We model the probabilistic relationships between the target disease node `AHD` and the patient's symptoms/clinical indicators.

In medical diagnostics, having heart disease (`AHD`) influences the likelihood of presenting specific symptoms (Age distribution, Sex, Chest Pain type, Maximum Heart Rate, Exercise Angina, and ST Depression Oldpeak).

In [ ]:
# Define the Directed Acyclic Graph (DAG) edges
edges = [
    ("AHD", "Age"),
    ("AHD", "Sex"),
    ("AHD", "ChestPain"),
    ("AHD", "MaxHR"),
    ("AHD", "ExAng"),
    ("AHD", "Oldpeak")
]

model = DiscreteBayesianNetwork(edges)
print("Bayesian Network graph nodes:", model.nodes())
print("Bayesian Network graph edges:", model.edges())

#### Visualize the Bayesian Network Graph

In [ ]:
plt.figure(figsize=(9, 6))
G = nx.DiGraph(model.edges())
pos = nx.spring_layout(G, seed=42)

# Differentiate target node vs symptom nodes
node_colors = ["#FF6B6B" if node == "AHD" else "#4D96FF" for node in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2500, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=11, font_weight="bold", font_color="white")
nx.draw_networkx_edges(G, pos, edge_color="#6c757d", arrows=True, arrowsize=20, width=2, connectionstyle="arc3,rad=0.1")

plt.title("Bayesian Network Structure for Heart Disease Diagnosis", fontsize=14, pad=15)
plt.axis("off")
plt.show()

### Step 6: Parameter Estimation (Learning CPDs)
We learn the **Conditional Probability Distributions (CPDs)** for each node from the training data using the `BayesianEstimator` with a `BDeu` (Bayesian Dirichlet equivalent uniform) prior to avoid zero-probability issues.

In [ ]:
# Parameter estimation using BayesianEstimator with BDeu prior
estimator = BayesianEstimator(model, train)
cpds = estimator.get_parameters(prior_type="BDeu", equivalent_sample_size=10)

# Add CPDs to the Bayesian Network model
model.add_cpds(*cpds)

# Check if model and CPDs are mathematically valid
is_valid = model.check_model()
print("Is the Bayesian Network Model valid? :", is_valid)

# Display sample CPD for ChestPain given AHD
print("\n--- Sample Learned CPD: P(ChestPain | AHD) ---")
print(model.get_cpds("ChestPain"))

### Step 7: Demonstration of Heart Patient Diagnosis (Probabilistic Inference)
The assignment specifically asks to **demonstrate the diagnosis of heart patients** using the model.
Using `VariableElimination`, we query $P(\text{AHD} \mid \text{Evidence})$ for different clinical scenarios:
1. **Patient 1 (High Risk Profile):** Male, high age, low max heart rate, asymptomatic chest pain, exercise angina, high ST depression.
2. **Patient 2 (Low Risk Profile):** Female, low age, high max heart rate, non-anginal pain, no exercise angina, low ST depression.
3. **Patient 3 (Partial Evidence):** Only Age and Chest Pain known — demonstrating Bayesian reasoning under incomplete information.

In [ ]:
infer = VariableElimination(model)

print("===========================================================")
print("DEMONSTRATION: PROBABILISTIC DIAGNOSIS OF HEART PATIENTS")
print("===========================================================")

# Case 1: High-Risk Patient Profile
high_risk_evidence = {
    "Age": 2,        # High age quantile
    "Sex": 1,        # Male
    "ChestPain": 0,  # Asymptomatic chest pain (highest risk category)
    "MaxHR": 0,      # Low maximum heart rate
    "ExAng": 1,      # Exercise induced angina present
    "Oldpeak": 2     # High ST depression
}
q_high = infer.query(variables=["AHD"], evidence=high_risk_evidence, show_progress=False)
print("\n[Case 1] High-Risk Patient Evidence:", high_risk_evidence)
print(f"-> P(AHD = 0 [No Heart Disease]): {q_high.values[0]:.4f} ({q_high.values[0]*100:.2f}%)")
print(f"-> P(AHD = 1 [Heart Disease]):    {q_high.values[1]:.4f} ({q_high.values[1]*100:.2f}%)")
print(f"-> Diagnosis: {'POSITIVE for Heart Disease' if q_high.values.argmax() == 1 else 'NEGATIVE'}")

# Case 2: Low-Risk Patient Profile
low_risk_evidence = {
    "Age": 0,        # Low age quantile
    "Sex": 0,        # Female
    "ChestPain": 1,  # Non-anginal pain
    "MaxHR": 2,      # High maximum heart rate
    "ExAng": 0,      # No exercise induced angina
    "Oldpeak": 0     # Low ST depression
}
q_low = infer.query(variables=["AHD"], evidence=low_risk_evidence, show_progress=False)
print("\n[Case 2] Low-Risk Patient Evidence:", low_risk_evidence)
print(f"-> P(AHD = 0 [No Heart Disease]): {q_low.values[0]:.4f} ({q_low.values[0]*100:.2f}%)")
print(f"-> P(AHD = 1 [Heart Disease]):    {q_low.values[1]:.4f} ({q_low.values[1]*100:.2f}%)")
print(f"-> Diagnosis: {'POSITIVE for Heart Disease' if q_low.values.argmax() == 1 else 'NEGATIVE'}")

# Case 3: Partial / Incomplete Clinical Evidence
partial_evidence = {
    "Age": 2,        # High age
    "ChestPain": 0   # Asymptomatic chest pain
}
q_partial = infer.query(variables=["AHD"], evidence=partial_evidence, show_progress=False)
print("\n[Case 3] Patient with Incomplete Evidence (Age & Chest Pain only):")
print(f"-> P(AHD = 1 [Heart Disease]): {q_partial.values[1]:.4f} ({q_partial.values[1]*100:.2f}%)")

### Step 8: Model Evaluation on Test Data
We evaluate the model on the unseen test set ($20\%$) to compute:
- **Accuracy**
- **Precision**
- **Recall**
- **Confusion Matrix**
- **Classification Report**

In [ ]:
evidence_cols = ["Age", "Sex", "ChestPain", "MaxHR", "ExAng", "Oldpeak"]
predictions = []

# Run probabilistic inference for each patient in the test set
for _, row in test.iterrows():
    evidence = {col: row[col] for col in evidence_cols}
    q = infer.query(variables=["AHD"], evidence=evidence, show_progress=False)
    predictions.append(int(q.values.argmax()))

y_test = test["AHD"].astype(int).values
predictions = np.array(predictions)

# Display first 15 comparisons
eval_df = pd.DataFrame({"Actual": y_test, "Predicted": predictions})
print("First 15 Test Predictions vs Actual Labels:")
print(eval_df.head(15).T)

In [ ]:
# Calculate Performance Metrics
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)
cm = confusion_matrix(y_test, predictions)

print("===========================================================")
print("           MODEL PERFORMANCE EVALUATION METRICS            ")
print("===========================================================")
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"Precision : {precision * 100:.2f}%")
print(f"Recall    : {recall * 100:.2f}%")
print("-----------------------------------------------------------")
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=["No Heart Disease (0)", "Heart Disease (1)"]))

# Visualize Confusion Matrix Heatmap
plt.figure(figsize=(6, 4.5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Disease (0)", "Heart Disease (1)"],
    yticklabels=["No Disease (0)", "Heart Disease (1)"]
)
plt.title("Confusion Matrix - Heart Disease Diagnosis", fontsize=12)
plt.xlabel("Predicted Label", fontsize=11)
plt.ylabel("True Label", fontsize=11)
plt.tight_layout()
plt.show()

### Step 9: Summary & Medical Insights
- **Bayesian Network Utility**: Unlike black-box machine learning algorithms, the Bayesian Network offers explainable probabilistic reasoning. Clinicians can see exactly how each piece of clinical evidence updates the posterior probability of heart disease.
- **Handling Incomplete Data**: The model can perform inference even when some diagnostic tests are unavailable or missing.
- **Clinical Relevance**: Key risk factors such as asymptomatic chest pain, older age, lower max heart rate, and exercise-induced angina significantly increase the posterior probability $P(\text{AHD}=1 \mid \text{Evidence})$, closely matching established cardiology guidelines.